# 📉 03_ Multi-Horizon Model Tournament & Selection
This notebook loads the fully consolidated, engineered dataset and executes the "Champion vs. Challenger" tournament across the 24h, 48h, and 72h horizons to determine the optimal production models.

### 📡 Block 1: Integrated Feature Loading & Temporal Splitting
In this section, we retrieve the finalized, fully integrated feature store rows containing our master feature set (engineered from your friend's logic, ByteCraft change-rates, and the Pearls Predictor architecture). We then apply a strict chronological time-series split to establish a training set (80%) and a testing set (20%), ensuring absolute temporal honesty without data leakage.

In [ ]:
all_data = []

page_size = 1000
start = 0

print("Fetching AQI feature dataset from Supabase...")

while True:

    response = (
        supabase
        .table("aqi_features")
        .select("*")
        .order("timestamp")
        .range(start, start + page_size - 1)
        .execute()
    )

    batch = response.data

    if not batch:
        break

    all_data.extend(batch)

    print(f"Collected {len(all_data)} rows...")

    start += page_size

# =============================================================================
# DATAFRAME
# =============================================================================

df_raw = pd.DataFrame(all_data)

# =============================================================================
# DATETIME
# =============================================================================

df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])

# =============================================================================
# SORT
# =============================================================================

df_raw = (
    df_raw
    .sort_values("timestamp")
    .drop_duplicates(subset=["timestamp"])
    .reset_index(drop=True)
)

# =============================================================================
# NUMERIC CONVERSION
# =============================================================================

numeric_cols = df_raw.columns.drop(["city", "timestamp"])

df_raw[numeric_cols] = (
    df_raw[numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
)

# =============================================================================
# CHECK TIME CONTINUITY
# =============================================================================

expected_hours = pd.date_range(
    start=df_raw["timestamp"].min(),
    end=df_raw["timestamp"].max(),
    freq="H"
)

print("\nExpected Hours:", len(expected_hours))
print("Actual Rows   :", len(df_raw))

# =============================================================================
# FINAL INFO
# =============================================================================

print(
    f"\n✅ Loaded {len(df_raw)} rows "
    f"| {df_raw['timestamp'].min()} "
    f"→ {df_raw['timestamp'].max()}"
)

print("\nColumns:")
print(list(df_raw.columns))

Fetching AQI feature dataset from Supabase...
Collected 1000 rows...
Collected 2000 rows...
Collected 3000 rows...
Collected 4000 rows...
Collected 5000 rows...
Collected 6000 rows...
Collected 7000 rows...
Collected 8000 rows...
Collected 9000 rows...
Collected 10000 rows...
Collected 11000 rows...
Collected 12000 rows...
Collected 13000 rows...
Collected 14000 rows...
Collected 15000 rows...
Collected 16000 rows...
Collected 17000 rows...
Collected 18000 rows...
Collected 19000 rows...
Collected 20000 rows...
Collected 21000 rows...
Collected 22000 rows...
Collected 23000 rows...
Collected 24000 rows...
Collected 25000 rows...
Collected 26000 rows...
Collected 27000 rows...
Collected 28000 rows...
Collected 29000 rows...
Collected 29400 rows...

Expected Hours: 29928
Actual Rows   : 29400

✅ Loaded 29400 rows | 2023-01-03 00:00:00 → 2026-06-02 23:00:00

Columns:
['id', 'timestamp', 'pm2_5', 'us_aqi', 'pm10', 'temperature', 'humidity', 'pressure', 'wind_speed', 'carbon_monoxide', 'nit

/tmp/ipykernel_15022/3446530068.py:68: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  expected_hours = pd.date_range(


### 🏆 Block 2: Multi-Horizon Model Tournament & Performance Summary
This block executes the automated model tournament loop across the 24h, 48h, and 72h forecasting horizons. It benchmarks our Champion (XGBoost) against our Challenger (Random Forest) using Mean Absolute Error (MAE) as the primary performance metrics, displaying a comprehensive summary table before promoting and saving the winning models.

In [ ]:
results_df = pd.DataFrame(results)

print("\n================ FINAL COMPARISON ================\n")

display(results_df)

# =============================================================================
# BEST MODEL PER HORIZON
# =============================================================================

best_models = results_df.loc[
    results_df.groupby("Horizon")["MAE"].idxmin()
]

print("\n================ BEST MODELS ================\n")

display(best_models)

# =============================================================================
# Find the winner for 24h
winner_24h = best_models[best_models["Horizon"] == "24h"].iloc[0]
winner_name = winner_24h["Model"]

print(f"\n================ TOP FEATURES (Winner: {winner_name}) ================\n")

# Check if the winner supports feature importance
if winner_name in trained_models:
    # Get the winning model object for 24h (index 0)
    winning_model = trained_models[winner_name][0]

    # Some models don't have feature_importances_ (like some older setups)
    if hasattr(winning_model, 'feature_importances_'):
        imp_df = pd.DataFrame({
            "feature": priority_features,
            "importance": winning_model.feature_importances_
        }).sort_values("importance", ascending=False)
        display(imp_df.head(30))
    else:
        print(f"Model {winner_name} does not support feature importance reporting.")
else:
    print(f"Feature importance not available for {winner_name}.")
# =============================================================================
# SAVE BEST MODELS
# =============================================================================

# =============================================================================
# DYNAMIC SAVE BEST MODELS (Tree-based & LSTM)
# =============================================================================
import joblib
import os

# Create a mapping for horizon indexes
horizon_map = {"24h": 0, "48h": 1, "72h": 2}

for _, row in best_models.iterrows():
    h = row["Horizon"]
    m_name = row["Model"]
    h_idx = horizon_map[h]

    # 1. Logic for Tree-based Models (Sklearn)
    if m_name in trained_models:
        best_obj = trained_models[m_name][h_idx]
        file_path = f"best_model_{h}.pkl"
        joblib.dump(best_obj, file_path)
        print(f"✅ Saved {m_name} (Sklearn) as {file_path}")

    # 2. Logic for LSTM Models (Keras)
    elif m_name == "LSTM":
        # Retrieve the specific LSTM model from your training loop
        # Note: 'model' in your loop is overwritten, so ensure you store
        # LSTMs in a list just like you did for tree models.
        best_obj = lstm_models[h_idx]
        file_path = f"best_model_{h}.keras"
        best_obj.save(file_path)
        print(f"✅ Saved LSTM (Keras) as {file_path}")

print("\n✅ All best models processed and saved.")


================ FINAL COMPARISON ================



,Model,Horizon,MAE,RMSE,R2
0,ExtraTrees,24h,8.658,11.983,0.336
1,ExtraTrees,48h,9.475,12.726,0.239
2,ExtraTrees,72h,9.694,12.754,0.246
3,RandomForest,24h,8.767,11.972,0.337
4,RandomForest,48h,9.756,12.934,0.214
5,RandomForest,72h,9.992,12.934,0.224
6,XGBoost,24h,8.923,12.419,0.286
7,XGBoost,48h,9.831,13.221,0.178
8,XGBoost,72h,10.203,13.496,0.155
9,LightGBM,24h,8.959,12.490,0.278



================ BEST MODELS ================



,Model,Horizon,MAE,RMSE,R2
0,ExtraTrees,24h,8.658,11.983,0.336
1,ExtraTrees,48h,9.475,12.726,0.239
2,ExtraTrees,72h,9.694,12.754,0.246



================ TOP FEATURES (Winner: ExtraTrees) ================



,feature,importance
8,month_cos,0.075792
38,high_pollution_flag,0.065181
10,season,0.035135
44,pm_ratio,0.034389
42,ozone,0.031696
2,pm25_log,0.030389
7,month_sin,0.028690
0,pm2_5,0.027536
11,aqi_lag_24h,0.025684
19,pm25_lag_24,0.025552


✅ Saved ExtraTrees (Sklearn) as best_model_24h.pkl
✅ Saved ExtraTrees (Sklearn) as best_model_48h.pkl
✅ Saved ExtraTrees (Sklearn) as best_model_72h.pkl

✅ All best models processed and saved.
